In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.utils import resample
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF, RationalQuadratic
import time
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.models.mlp import MLP, create_mlp_pytorch, EarlyStopping
from src.models.resnet import ResBlock, ResNet
import torch.optim as optim

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No no

In [2]:
descriptor_df = pd.read_csv('data/freesolv/external_descriptors.csv')
smiles_df = pd.read_csv('data/freesolv/smiles.csv')
learned_descriptors_df = pd.read_csv('data/freesolv/learned_predictors_0.csv')
df = pd.concat([learned_descriptors_df, descriptor_df, smiles_df], axis=1)

In [3]:
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,209,210,211,212,213,214,215,y,w,ids
0,0.005919,0.0,0.000000,0.0,0.0,0.0,0.0,0.009849,0.030713,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.107729,1.0,C(CCl)OCCCl
1,0.016760,0.0,0.004031,0.0,0.0,0.0,0.0,0.002265,0.022694,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.014446,1.0,C(Cl)(Cl)(Cl)Cl
2,0.006240,0.0,0.010199,0.0,0.0,0.0,0.0,0.007726,0.026535,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199502,1.0,CC(C)CC(=O)C
3,0.007206,0.0,0.005383,0.0,0.0,0.0,0.0,0.005392,0.038531,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.449453,1.0,CCCCO[N+](=O)[O-]
4,0.003805,0.0,0.006889,0.0,0.0,0.0,0.0,0.004972,0.007662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.574428,1.0,CSC


In [4]:
df = df.drop(columns='ids')
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,208,209,210,211,212,213,214,215,y,w
0,0.005919,0.0,0.000000,0.0,0.0,0.0,0.0,0.009849,0.030713,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.107729,1.0
1,0.016760,0.0,0.004031,0.0,0.0,0.0,0.0,0.002265,0.022694,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.014446,1.0
2,0.006240,0.0,0.010199,0.0,0.0,0.0,0.0,0.007726,0.026535,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199502,1.0
3,0.007206,0.0,0.005383,0.0,0.0,0.0,0.0,0.005392,0.038531,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.449453,1.0
4,0.003805,0.0,0.006889,0.0,0.0,0.0,0.0,0.004972,0.007662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.574428,1.0


In [5]:
df['w'].unique()

array([1.])

In [6]:
df = df.drop(columns='w')
df.head()

,fp_0,fp_1,fp_2,fp_3,fp_4,fp_5,fp_6,fp_7,fp_8,fp_9,...,207,208,209,210,211,212,213,214,215,y
0,0.005919,0.0,0.000000,0.0,0.0,0.0,0.0,0.009849,0.030713,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.107729
1,0.016760,0.0,0.004031,0.0,0.0,0.0,0.0,0.002265,0.022694,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.014446
2,0.006240,0.0,0.010199,0.0,0.0,0.0,0.0,0.007726,0.026535,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.199502
3,0.007206,0.0,0.005383,0.0,0.0,0.0,0.0,0.005392,0.038531,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.449453
4,0.003805,0.0,0.006889,0.0,0.0,0.0,0.0,0.004972,0.007662,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.574428


In [7]:
X = df.drop(columns=['y'])
y = df['y']

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

In [9]:
models_df = pd.DataFrame(columns=[
    'index', 'model_type', 'hyperparams', 'rmse'
])
models_list = []
config_id = 0
base_seed = 42

In [10]:
for i in range(4):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KNeighborsRegressor()
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': {},
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)
 

{'index': 0, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.49996243256791545}
{'index': 1, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.5222536143260095}
{'index': 2, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.38679363559629854}
{'index': 3, 'model_type': 'KNeighborsRegressor', 'hyperparams': {}, 'rmse': 0.5410009043054692}


In [11]:
param_grid = {
    'alpha': [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = Lasso(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 4, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.001}, 'rmse': 0.7347778409931316}
{'index': 5, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.002}, 'rmse': 0.26512792887063996}
{'index': 6, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.005}, 'rmse': 0.8263450318347525}
{'index': 7, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.01}, 'rmse': 1.854749033145995}
{'index': 8, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.02}, 'rmse': 0.1928459522136429}
{'index': 9, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.05}, 'rmse': 0.2536403866614325}
{'index': 10, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.1}, 'rmse': 0.31321846948737103}
{'index': 11, 'model_type': 'Lasso', 'hyperparams': {'alpha': 0.2}, 'rmse': 0.41072672943340344}


In [12]:
param_grid = {
    'alpha': [1, 2, 3, 5, 7, 10, 15, 20, 30, 50]
}

for params in ParameterGrid(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = Ridge(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 12, 'model_type': 'Ridge', 'hyperparams': {'alpha': 1}, 'rmse': 0.2960534197447393}
{'index': 13, 'model_type': 'Ridge', 'hyperparams': {'alpha': 2}, 'rmse': 0.29225501038278595}
{'index': 14, 'model_type': 'Ridge', 'hyperparams': {'alpha': 3}, 'rmse': 0.2710033269557431}
{'index': 15, 'model_type': 'Ridge', 'hyperparams': {'alpha': 5}, 'rmse': 1.1887551574438189}
{'index': 16, 'model_type': 'Ridge', 'hyperparams': {'alpha': 7}, 'rmse': 11.555681826339478}
{'index': 17, 'model_type': 'Ridge', 'hyperparams': {'alpha': 10}, 'rmse': 1.1231994464637927}
{'index': 18, 'model_type': 'Ridge', 'hyperparams': {'alpha': 15}, 'rmse': 0.5259882149774259}
{'index': 19, 'model_type': 'Ridge', 'hyperparams': {'alpha': 20}, 'rmse': 0.422945773315146}
{'index': 20, 'model_type': 'Ridge', 'hyperparams': {'alpha': 30}, 'rmse': 0.653360916124169}
{'index': 21, 'model_type': 'Ridge', 'hyperparams': {'alpha': 50}, 'rmse': 0.4628901739753788}


In [13]:
param_grid = [
    # Polynomial kernel (3rd degree)
    {
        'kernel': ['poly'],
        'alpha': [0.1, 1, 10],
        'degree': [3],
    },
    # RBF kernel
    {
        'kernel': ['rbf'],
        'alpha': [0.1, 1, 10],
    }
]

for idx, params in enumerate(ParameterGrid(param_grid)):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = KernelRidge(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 22, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 0.1, 'degree': 3, 'kernel': 'poly'}, 'rmse': 2.600431090152945}
{'index': 23, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 1, 'degree': 3, 'kernel': 'poly'}, 'rmse': 1.622763772072651}
{'index': 24, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 10, 'degree': 3, 'kernel': 'poly'}, 'rmse': 693.0348927790754}
{'index': 25, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 0.1, 'kernel': 'rbf'}, 'rmse': 0.26610191378502307}
{'index': 26, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 1, 'kernel': 'rbf'}, 'rmse': 0.2501666712953179}
{'index': 27, 'model_type': 'KernelRidge', 'hyperparams': {'alpha': 10, 'kernel': 'rbf'}, 'rmse': 0.4384001137834936}


In [14]:
param_grid = [
    {'max_depth': [3], 'n_estimators': [10]},
    {'max_depth': [5], 'n_estimators': [10]},
    {'max_depth': [5], 'n_estimators': [20]},
    {'max_depth': [5], 'n_estimators': [50]},
    {'max_depth': [8], 'n_estimators': [50]},
    {'max_depth': [8], 'n_estimators': [100]},
    {'max_depth': [5], 'n_estimators': [100]},
    {'max_depth': [8], 'n_estimators': [100]},
    {'max_depth': [16], 'n_estimators': [200]},
    {'max_depth': [32], 'n_estimators': [1000]}
]
for idx, params in enumerate(ParameterGrid(param_grid)):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    model = RandomForestRegressor(**params)
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)

    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)

{'index': 28, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 10}, 'rmse': 0.3212455108749685}
{'index': 29, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 10}, 'rmse': 0.315533500178486}
{'index': 30, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 20}, 'rmse': 0.2735783325803757}
{'index': 31, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 50}, 'rmse': 0.2758995581148188}
{'index': 32, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 8, 'n_estimators': 50}, 'rmse': 0.23195875773799232}
{'index': 33, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 8, 'n_estimators': 100}, 'rmse': 0.24158783043054313}
{'index': 34, 'model_type': 'RandomForestRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 100}, 'rmse': 0.25194093331547823}
{'index': 35, 'model_type': 'RandomForestRegressor', 'hype

In [15]:
hyperparams_list = [
    {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.1},
    {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 200, 'learning_rate': 0.2},
    {'max_depth': 5, 'n_estimators': 500, 'learning_rate': 0.1},
    {'max_depth': 10, 'n_estimators': 500, 'learning_rate': 0.01},
    {'max_depth': 10, 'n_estimators': 1000, 'learning_rate': 0.1},
    {'max_depth': 32, 'n_estimators': 2000, 'learning_rate': 0.1}
]

for idx, params in enumerate(hyperparams_list):

    seed = base_seed + config_id

    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    start_time = time.time()
    model = XGBRegressor(
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        n_jobs=-1,
    )

    model.fit(X_boot_scaled, y_boot)

    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    
    rmse = root_mean_squared_error(y_val, y_val_pred)
    training_time = time.time() - start_time
    
    print(f'Training time: {training_time}')
    
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    
    config_id += 1
    
    print(model_metadata)
    
    models_list.append(model_metadata)

Training time: 0.16964197158813477
{'index': 38, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.1}, 'rmse': 0.27926179644011795}
Training time: 0.24590206146240234
{'index': 39, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.1}, 'rmse': 0.2661732314680289}
Training time: 0.21074867248535156
{'index': 40, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 3, 'n_estimators': 200, 'learning_rate': 0.1}, 'rmse': 0.2245320919269314}
Training time: 0.5278887748718262
{'index': 41, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 10, 'n_estimators': 200, 'learning_rate': 0.2}, 'rmse': 0.2985225843559641}
Training time: 0.7396421432495117
{'index': 42, 'model_type': 'XGBRegressor', 'hyperparams': {'max_depth': 5, 'n_estimators': 500, 'learning_rate': 0.1}, 'rmse': 0.2624869346979912}
Training time: 4.73390531539917
{'index': 43, 'model_type': 'XGBRegressor', 'hyperpar

In [16]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, RBF, RationalQuadratic

param_grid = [
    # Matern kernel models
    {
        'kernel': Matern(),  
    },
    {
        'kernel': Matern(),  
    },
    # Quadratic (RBF) kernel models  
    {
        'kernel': RBF(),  
    },
    {
        'kernel': RBF(),
    }
]

for idx, params in enumerate(param_grid):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    start_time = time.time()
    model = GaussianProcessRegressor(kernel=params['kernel'])
    model.fit(X_boot_scaled, y_boot)
    X_val_scaled = scaler.transform(X_val)
    y_val_pred = model.predict(X_val_scaled)
    rmse = root_mean_squared_error(y_val, y_val_pred)
    training_time = time.time() - start_time
    print(f'Training time: {training_time}')
    model_metadata = {
        'index': config_id,
        'model_type': type(model).__name__,
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    
    print(model_metadata)
    models_list.append(model_metadata)


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 4 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


Training time: 1.7045562267303467
{'index': 46, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'rmse': 0.18826171894855961}
Training time: 0.5200386047363281
{'index': 47, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': Matern(length_scale=1, nu=1.5)}, 'rmse': 0.18509717636453746}
Training time: 1.3836884498596191
{'index': 48, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'rmse': 0.2579589574628045}
Training time: 0.5759625434875488
{'index': 49, 'model_type': 'GaussianProcessRegressor', 'hyperparams': {'kernel': RBF(length_scale=1)}, 'rmse': 0.17649267088808826}


In [17]:
import torch.nn as nn

In [18]:
hyperparams_list = [
    {'n_layers': 2, 'layer_size': 5, 'lr': 0.01, 'l2_reg': 0.01},
    {'n_layers': 2, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.001, 'l2_reg': 0.1},
    {'n_layers': 3, 'layer_size': 10, 'lr': 0.0005, 'l2_reg': 0.1}
]
for idx, params in enumerate(hyperparams_list):
    
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)

    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)

    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    # Create MLP
    model, optimizer = create_mlp_pytorch(
        params['input_dim'],
        params['output_dim'], 
        n_layers=params['n_layers'],
        layer_size=params['layer_size'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])
    
    # Training code would go here
    # model.train() ... etc.
    # Basic EarlyStopping for 1000 epochs
    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    rmse = root_mean_squared_error(all_targets, all_predictions)
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'MLP_PyTorch',
        'hyperparams': params,
        'rmse': rmse
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 1.0732, Val Loss: 0.5000
Validation loss decreased (inf -> 0.499981). Saving model...


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 2/1000, Train Loss: 1.1559, Val Loss: 0.4616
Validation loss decreased (0.499981 -> 0.461623). Saving model...
Epoch 3/1000, Train Loss: 1.0311, Val Loss: 0.4578
Validation loss decreased (0.461623 -> 0.457849). Saving model...
Epoch 4/1000, Train Loss: 0.9899, Val Loss: 0.4597
EarlyStopping counter: 1 out of 20
Epoch 5/1000, Train Loss: 0.9939, Val Loss: 0.4584
EarlyStopping counter: 2 out of 20
Epoch 6/1000, Train Loss: 0.9831, Val Loss: 0.4584
EarlyStopping counter: 3 out of 20
Epoch 7/1000, Train Loss: 1.0214, Val Loss: 0.4570
EarlyStopping counter: 4 out of 20
Epoch 8/1000, Train Loss: 1.0780, Val Loss: 0.4625
EarlyStopping counter: 5 out of 20
Epoch 9/1000, Train Loss: 1.0650, Val Loss: 0.4672
EarlyStopping counter: 6 out of 20
Epoch 10/1000, Train Loss: 1.0271, Val Loss: 0.4677
EarlyStopping counter: 7 out of 20
Epoch 11/1000, Train Loss: 0.9915, Val Loss: 0.4624
EarlyStopping counter: 8 out of 20
Epoch 12/1000, Train Loss: 1.0449, Val Loss: 0.4605
EarlyStopping counter: 9

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 4/1000, Train Loss: 0.9131, Val Loss: 0.4622
EarlyStopping counter: 2 out of 20
Epoch 5/1000, Train Loss: 0.9093, Val Loss: 0.4600
Validation loss decreased (0.463118 -> 0.459987). Saving model...
Epoch 6/1000, Train Loss: 0.9535, Val Loss: 0.4603
EarlyStopping counter: 1 out of 20
Epoch 7/1000, Train Loss: 1.0404, Val Loss: 0.4590
Validation loss decreased (0.459987 -> 0.458968). Saving model...
Epoch 8/1000, Train Loss: 0.9143, Val Loss: 0.4609
EarlyStopping counter: 1 out of 20
Epoch 9/1000, Train Loss: 0.9086, Val Loss: 0.4583
EarlyStopping counter: 2 out of 20
Epoch 10/1000, Train Loss: 0.9133, Val Loss: 0.4581
EarlyStopping counter: 3 out of 20
Epoch 11/1000, Train Loss: 0.9081, Val Loss: 0.4589
EarlyStopping counter: 4 out of 20
Epoch 12/1000, Train Loss: 0.9188, Val Loss: 0.4592
EarlyStopping counter: 5 out of 20
Epoch 13/1000, Train Loss: 0.9426, Val Loss: 0.4602
EarlyStopping counter: 6 out of 20
Epoch 14/1000, Train Loss: 0.9302, Val Loss: 0.4589
EarlyStopping counter:

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 7/1000, Train Loss: 1.0987, Val Loss: 0.4834
EarlyStopping counter: 3 out of 20
Epoch 8/1000, Train Loss: 0.9729, Val Loss: 0.4676
Validation loss decreased (0.468822 -> 0.467597). Saving model...
Epoch 9/1000, Train Loss: 0.9234, Val Loss: 0.4638
Validation loss decreased (0.467597 -> 0.463842). Saving model...
Epoch 10/1000, Train Loss: 0.8917, Val Loss: 0.4657
EarlyStopping counter: 1 out of 20
Epoch 11/1000, Train Loss: 0.8807, Val Loss: 0.4665
EarlyStopping counter: 2 out of 20
Epoch 12/1000, Train Loss: 0.8761, Val Loss: 0.4646
EarlyStopping counter: 3 out of 20
Epoch 13/1000, Train Loss: 0.8877, Val Loss: 0.4634
EarlyStopping counter: 4 out of 20
Epoch 14/1000, Train Loss: 0.8749, Val Loss: 0.4634
EarlyStopping counter: 5 out of 20
Epoch 15/1000, Train Loss: 0.8762, Val Loss: 0.4639
EarlyStopping counter: 6 out of 20
Epoch 16/1000, Train Loss: 0.9015, Val Loss: 0.4635
EarlyStopping counter: 7 out of 20
Epoch 17/1000, Train Loss: 0.9805, Val Loss: 0.4647
EarlyStopping count

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 2/1000, Train Loss: 1.0827, Val Loss: 0.5596
Validation loss decreased (0.604098 -> 0.559620). Saving model...
Epoch 3/1000, Train Loss: 1.0262, Val Loss: 0.5458
Validation loss decreased (0.559620 -> 0.545767). Saving model...
Epoch 4/1000, Train Loss: 0.9620, Val Loss: 0.5374
Validation loss decreased (0.545767 -> 0.537428). Saving model...
Epoch 5/1000, Train Loss: 0.9705, Val Loss: 0.5277
Validation loss decreased (0.537428 -> 0.527750). Saving model...
Epoch 6/1000, Train Loss: 0.9484, Val Loss: 0.5226
Validation loss decreased (0.527750 -> 0.522567). Saving model...
Epoch 7/1000, Train Loss: 0.9733, Val Loss: 0.5164
Validation loss decreased (0.522567 -> 0.516395). Saving model...
Epoch 8/1000, Train Loss: 0.9857, Val Loss: 0.5118
Validation loss decreased (0.516395 -> 0.511828). Saving model...
Epoch 9/1000, Train Loss: 0.9403, Val Loss: 0.5067
Validation loss decreased (0.511828 -> 0.506720). Saving model...
Epoch 10/1000, Train Loss: 0.9613, Val Loss: 0.5035
Validation l

In [19]:
def create_mlp_resnet(input_dim, output_dim, block_dim, hidden_dim, num_blocks, lr, l2_reg):
    model = ResNet(
        input_dim = input_dim,
        output_dim = output_dim,
        num_blocks = num_blocks,
        hidden_dim = hidden_dim,
        block_dim = block_dim
    )
    optimizer = optim.Adam(model.parameters(), lr, weight_decay=l2_reg)

    return model, optimizer

In [20]:
specified_configs = [
    # D block D hidden N blocks Learning rate L2 regularisation
    {'block_dim': 16, 'hidden_dim': 8, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.01},
    {'block_dim': 32, 'hidden_dim': 16, 'num_blocks': 2, 'lr': 0.01, 'l2_reg': 0.1},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.01},
    {'block_dim': 64, 'hidden_dim': 32, 'num_blocks': 3, 'lr': 0.001, 'l2_reg': 0.1},
]

for idx, params in enumerate(specified_configs):
    seed = base_seed + config_id
    X_boot, y_boot = resample(
    X_train, y_train,
    replace=True,
    n_samples=len(X_train),
    random_state=seed)
    params.update({
    'input_dim': X_boot.shape[1],
    'output_dim': 1
})
    scaler = StandardScaler()
    X_boot_scaled = scaler.fit_transform(X_boot)
    X_boot_tensor = torch.FloatTensor(X_boot_scaled)
    y_boot_tensor = torch.FloatTensor(y_boot.values)
    train_dataset = TensorDataset(X_boot_tensor, y_boot_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    X_val_scaled = scaler.transform(X_val)
    X_val_tensor = torch.FloatTensor(X_val_scaled)
    y_val_tensor = torch.FloatTensor(y_val.values)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32)

    model, optimizer = create_mlp_resnet(
        input_dim = params['input_dim'], 
        output_dim = params['output_dim'],
        num_blocks=params['num_blocks'],
        hidden_dim=params['hidden_dim'],
        block_dim=params['block_dim'],
        lr=params['lr'],
        l2_reg=params['l2_reg'])

    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()

    early_stopping = EarlyStopping(patience=20, min_delta=0.001, verbose=True)
    criterion = nn.MSELoss()
  
    train_losses = []
    val_losses = []
    
    for epoch in range(1000):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                val_loss += criterion(output, target).item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{1000}, '
                f'Train Loss: {avg_train_loss:.4f}, '
                f'Val Loss: {avg_val_loss:.4f}')
        
        # Early stopping check
        if early_stopping(avg_val_loss, model):
            print("Early stopping triggered. Restoring best model...")
            # Restore the best model
            model.load_state_dict(early_stopping.best_model_state)
            break
    
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(val_loader):
            predictions = model(data)
            
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(target.numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate RMSE
    rmse = root_mean_squared_error(all_targets, all_predictions)
    
    # Store metadata
    model_metadata = {
        'index': config_id,
        'model_type': 'ResNet',
        'hyperparams': None,
        'rmse': rmse
    }
    config_id += 1
    print(model_metadata)
    models_list.append(model_metadata)

Epoch 1/1000, Train Loss: 1.1259, Val Loss: 0.5871
Validation loss decreased (inf -> 0.587057). Saving model...
Epoch 2/1000, Train Loss: 1.0861, Val Loss: 0.6002
EarlyStopping counter: 1 out of 20
Epoch 3/1000, Train Loss: 1.0753, Val Loss: 0.4756
Validation loss decreased (0.587057 -> 0.475588). Saving model...
Epoch 4/1000, Train Loss: 1.1239, Val Loss: 0.4916
EarlyStopping counter: 1 out of 20
Epoch 5/1000, Train Loss: 0.9680, Val Loss: 0.5254
EarlyStopping counter: 2 out of 20


/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 6/1000, Train Loss: 1.0026, Val Loss: 0.4895
EarlyStopping counter: 3 out of 20
Epoch 7/1000, Train Loss: 1.0180, Val Loss: 0.5715
EarlyStopping counter: 4 out of 20
Epoch 8/1000, Train Loss: 0.9442, Val Loss: 0.4885
EarlyStopping counter: 5 out of 20
Epoch 9/1000, Train Loss: 0.9408, Val Loss: 0.4578
Validation loss decreased (0.475588 -> 0.457821). Saving model...
Epoch 10/1000, Train Loss: 0.9379, Val Loss: 0.4661
EarlyStopping counter: 1 out of 20
Epoch 11/1000, Train Loss: 1.1183, Val Loss: 0.4719
EarlyStopping counter: 2 out of 20
Epoch 12/1000, Train Loss: 0.9779, Val Loss: 0.5233
EarlyStopping counter: 3 out of 20
Epoch 13/1000, Train Loss: 1.0668, Val Loss: 0.4595
EarlyStopping counter: 4 out of 20
Epoch 14/1000, Train Loss: 0.9390, Val Loss: 0.4612
EarlyStopping counter: 5 out of 20
Epoch 15/1000, Train Loss: 0.9366, Val Loss: 0.4605
EarlyStopping counter: 6 out of 20
Epoch 16/1000, Train Loss: 0.9396, Val Loss: 0.4590
EarlyStopping counter: 7 out of 20
Epoch 17/1000, T

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 9/1000, Train Loss: 1.2386, Val Loss: 0.4617
EarlyStopping counter: 2 out of 20
Epoch 10/1000, Train Loss: 1.2187, Val Loss: 0.4612
EarlyStopping counter: 3 out of 20
Epoch 11/1000, Train Loss: 1.2122, Val Loss: 0.4727
EarlyStopping counter: 4 out of 20
Epoch 12/1000, Train Loss: 1.2182, Val Loss: 0.4721
EarlyStopping counter: 5 out of 20
Epoch 13/1000, Train Loss: 2.7605, Val Loss: 0.4581
Validation loss decreased (0.461057 -> 0.458083). Saving model...
Epoch 14/1000, Train Loss: 1.2302, Val Loss: 0.4628
EarlyStopping counter: 1 out of 20
Epoch 15/1000, Train Loss: 1.2249, Val Loss: 0.4600
EarlyStopping counter: 2 out of 20
Epoch 16/1000, Train Loss: 1.3174, Val Loss: 0.4576
EarlyStopping counter: 3 out of 20
Epoch 17/1000, Train Loss: 1.2205, Val Loss: 0.4578
EarlyStopping counter: 4 out of 20
Epoch 18/1000, Train Loss: 1.2983, Val Loss: 0.4735
EarlyStopping counter: 5 out of 20
Epoch 19/1000, Train Loss: 1.2187, Val Loss: 0.4662
EarlyStopping counter: 6 out of 20
Epoch 20/1000

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 6/1000, Train Loss: 1.1074, Val Loss: 5.2606
EarlyStopping counter: 5 out of 20
Epoch 7/1000, Train Loss: 1.1038, Val Loss: 1.1927
EarlyStopping counter: 6 out of 20
Epoch 8/1000, Train Loss: 1.2034, Val Loss: 1.0252
EarlyStopping counter: 7 out of 20
Epoch 9/1000, Train Loss: 1.1055, Val Loss: 1.1420
EarlyStopping counter: 8 out of 20
Epoch 10/1000, Train Loss: 1.1136, Val Loss: 2.3466
EarlyStopping counter: 9 out of 20
Epoch 11/1000, Train Loss: 1.1295, Val Loss: 0.8711
EarlyStopping counter: 10 out of 20
Epoch 12/1000, Train Loss: 1.2151, Val Loss: 0.5718
Validation loss decreased (0.640294 -> 0.571754). Saving model...
Epoch 13/1000, Train Loss: 1.1104, Val Loss: 0.5740
EarlyStopping counter: 1 out of 20
Epoch 14/1000, Train Loss: 1.1714, Val Loss: 1.0002
EarlyStopping counter: 2 out of 20
Epoch 15/1000, Train Loss: 1.1302, Val Loss: 3.6765
EarlyStopping counter: 3 out of 20
Epoch 16/1000, Train Loss: 1.0972, Val Loss: 2.2596
EarlyStopping counter: 4 out of 20
Epoch 17/1000, 

/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/prati/miniconda3/envs/chemprop_env/lib/python3.11/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 3/1000, Train Loss: 1.0258, Val Loss: 0.5144
EarlyStopping counter: 1 out of 20
Epoch 4/1000, Train Loss: 1.0620, Val Loss: 0.4688
EarlyStopping counter: 2 out of 20
Epoch 5/1000, Train Loss: 1.0344, Val Loss: 0.4731
EarlyStopping counter: 3 out of 20
Epoch 6/1000, Train Loss: 1.0509, Val Loss: 0.4969
EarlyStopping counter: 4 out of 20
Epoch 7/1000, Train Loss: 1.0534, Val Loss: 0.4767
EarlyStopping counter: 5 out of 20
Epoch 8/1000, Train Loss: 1.0395, Val Loss: 0.4754
EarlyStopping counter: 6 out of 20
Epoch 9/1000, Train Loss: 1.1424, Val Loss: 0.4746
EarlyStopping counter: 7 out of 20
Epoch 10/1000, Train Loss: 1.0478, Val Loss: 0.4686
EarlyStopping counter: 8 out of 20
Epoch 11/1000, Train Loss: 1.0662, Val Loss: 0.4627
Validation loss decreased (0.464036 -> 0.462702). Saving model...
Epoch 12/1000, Train Loss: 1.1206, Val Loss: 0.4642
EarlyStopping counter: 1 out of 20
Epoch 13/1000, Train Loss: 1.1732, Val Loss: 0.4812
EarlyStopping counter: 2 out of 20
Epoch 14/1000, Trai

In [21]:
models_list

[{'index': 0,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.49996243256791545},
 {'index': 1,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.5222536143260095},
 {'index': 2,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.38679363559629854},
 {'index': 3,
  'model_type': 'KNeighborsRegressor',
  'hyperparams': {},
  'rmse': 0.5410009043054692},
 {'index': 4,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.001},
  'rmse': 0.7347778409931316},
 {'index': 5,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.002},
  'rmse': 0.26512792887063996},
 {'index': 6,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.005},
  'rmse': 0.8263450318347525},
 {'index': 7,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.01},
  'rmse': 1.854749033145995},
 {'index': 8,
  'model_type': 'Lasso',
  'hyperparams': {'alpha': 0.02},
  'rmse': 0.1928459522136429},
 {'index': 9,
  'model_type': 'Lasso',
  'hyperparams':

In [22]:
models_df = pd.DataFrame(models_list)
models_df = models_df.sort_values('rmse', ascending=True).reset_index(drop=True)
final_models = models_df.head(10)
final_models

,index,model_type,hyperparams,rmse
0,49,GaussianProcessRegressor,{'kernel': RBF(length_scale=1)},0.176493
1,47,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.185097
2,46,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.188262
3,8,Lasso,{'alpha': 0.02},0.192846
4,40,XGBRegressor,"{'max_depth': 3, 'n_estimators': 200, 'learnin...",0.224532
5,32,RandomForestRegressor,"{'max_depth': 8, 'n_estimators': 50}",0.231959
6,33,RandomForestRegressor,"{'max_depth': 8, 'n_estimators': 100}",0.241588
7,26,KernelRidge,"{'alpha': 1, 'kernel': 'rbf'}",0.250167
8,34,RandomForestRegressor,"{'max_depth': 5, 'n_estimators': 100}",0.251941
9,9,Lasso,{'alpha': 0.05},0.253640


In [23]:
def softmax(series):
    exp_x = np.exp(series - series.max())
    return exp_x / exp_x.sum()

In [24]:
final_models['weights'] = softmax(-final_models['rmse'])
final_models

/tmp/ipykernel_384108/2114309755.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_models['weights'] = softmax(-final_models['rmse'])


,index,model_type,hyperparams,rmse,weights
0,49,GaussianProcessRegressor,{'kernel': RBF(length_scale=1)},0.176493,0.104366
1,47,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.185097,0.103472
2,46,GaussianProcessRegressor,"{'kernel': Matern(length_scale=1, nu=1.5)}",0.188262,0.103145
3,8,Lasso,{'alpha': 0.02},0.192846,0.102673
4,40,XGBRegressor,"{'max_depth': 3, 'n_estimators': 200, 'learnin...",0.224532,0.099471
5,32,RandomForestRegressor,"{'max_depth': 8, 'n_estimators': 50}",0.231959,0.098735
6,33,RandomForestRegressor,"{'max_depth': 8, 'n_estimators': 100}",0.241588,0.097788
7,26,KernelRidge,"{'alpha': 1, 'kernel': 'rbf'}",0.250167,0.096953
8,34,RandomForestRegressor,"{'max_depth': 5, 'n_estimators': 100}",0.251941,0.096781
9,9,Lasso,{'alpha': 0.05},0.253640,0.096617


In [25]:
rmse = sum(final_models['rmse']*final_models['weights'])
rmse

0.2187951150946016